## Exploratory Data Analysis and Cleanup for Children's Books

In [1]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

/Users/navyajain/Downloads/build/agentic-book-recommendation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DetectorFactory.seed = 42

In [3]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

True

In [4]:
#print(children_path)

In [5]:
comic_df = pd.read_json('../data/books/goodreads_books_fantasy_paranormal.json', lines=True)

In [6]:
comic_df.shape

(258585, 29)

In [7]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,7,[189911],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,false,4.03,,"[19997, 828466, 1569323, 425389, 1176674, 2627...",Omnibus book club edition containing the Ladie...,Hardcover,https://www.goodreads.com/book/show/7327624-th...,"[{'author_id': '10333', 'role': ''}]","Nelson Doubleday, Inc.",600,,,,Book Club Edition,1987,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,1934876569,6,[151854],US,,"[{'count': '515', 'name': 'to-read'}, {'count'...",,false,4.22,,"[948696, 439885, 274955, 12978730, 372986, 216...","To Kara's astonishment, she discovers that a p...",Paperback,https://www.goodreads.com/book/show/6066812-al...,"[{'author_id': '19158', 'role': ''}]",Seven Seas,216,3,9781934876565,3,,2009,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...
2,,60,[1052227],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,true,4.33,B01NCIKAQX,[],,,https://www.goodreads.com/book/show/33394837-t...,"[{'author_id': '242185', 'role': ''}]",,318,,,,,,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269,54143148,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
3,,1,[147734],US,,"[{'count': '1057', 'name': 'to-read'}, {'count...",B0056A00P4,true,4.04,B0056A00P4,"[519546, 1295074, 21407416]",This is the final tale in the bestselling auth...,,https://www.goodreads.com/book/show/12182387-t...,"[{'author_id': '50873', 'role': ''}, {'author_...",,,,,,,,https://www.goodreads.com/book/show/12182387-t...,https://s.gr-assets.com/assets/nophoto/book/11...,12182387,4,285263,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)"
4,,21,[811663],US,en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",B01BLJGA9S,true,4.23,B01BLJGA9S,"[25515353, 20483269, 25650829, 18913492, 22578...",,,https://www.goodreads.com/book/show/29074693-p...,"[{'author_id': '5360266', 'role': ''}]",,,,,,,,https://www.goodreads.com/book/show/29074693-p...,https://s.gr-assets.com/assets/nophoto/book/11...,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)"


In [8]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [9]:
df.shape

(258585, 29)

In [10]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [11]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [12]:
df.shape

(258585, 26)

In [13]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ...",[189911],US,eng,,B00071IKUY,,,8948723,7327624,7,4.03,140,"[{'count': '58', 'name': 'to-read'}, {'count':...",false,Hardcover,600,"Nelson Doubleday, Inc.",,,1987,Book Club Edition,Omnibus book club edition containing the Ladie...,"[{'author_id': '10333', 'role': ''}]","[19997, 828466, 1569323, 425389, 1176674, 2627..."
1,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],US,,1934876569,,,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",false,Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216..."
2,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2),[1052227],US,eng,,B01NCIKAQX,B01NCIKAQX,,54143148,33394837,60,4.33,269,"[{'count': '54', 'name': 'currently-reading'},...",true,,318,,,,,,,"[{'author_id': '242185', 'role': ''}]",[]
3,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)",[147734],US,,,B0056A00P4,B0056A00P4,,285263,12182387,1,4.04,4,"[{'count': '1057', 'name': 'to-read'}, {'count...",true,,,,,,,,This is the final tale in the bestselling auth...,"[{'author_id': '50873', 'role': ''}, {'author_...","[519546, 1295074, 21407416]"
4,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)",[811663],US,en-US,,B01BLJGA9S,B01BLJGA9S,,46079519,29074693,21,4.23,149,"[{'count': '598', 'name': 'to-read'}, {'count'...",true,,,,,,,,,"[{'author_id': '5360266', 'role': ''}]","[25515353, 20483269, 25650829, 18913492, 22578..."


In [14]:
df['clean_title'] = df['title_without_series']

In [15]:
df.shape

(258585, 27)

### Detecting languages

In [16]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [17]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 70664/70664 [03:50<00:00, 306.40it/s]


In [18]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ...",[189911],US,eng,,B00071IKUY,,,8948723,7327624,7,4.03,140,"[{'count': '58', 'name': 'to-read'}, {'count':...",false,Hardcover,600,"Nelson Doubleday, Inc.",,,1987,Book Club Edition,Omnibus book club edition containing the Ladie...,"[{'author_id': '10333', 'role': ''}]","[19997, 828466, 1569323, 425389, 1176674, 2627...","The Unschooled Wizard (Sun Wolf and Starhawk, ...",eng,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",nan
1,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],US,,1934876569,,,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",false,Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216...",All's Fairy in Love and War (Avalon: Web of Ma...,eng,All's Fairy in Love and War (Avalon: Web of Ma...,eng
2,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2),[1052227],US,eng,,B01NCIKAQX,B01NCIKAQX,,54143148,33394837,60,4.33,269,"[{'count': '54', 'name': 'currently-reading'},...",true,,318,,,,,,,"[{'author_id': '242185', 'role': ''}]",[],The House of Memory (Pluto's Snitch #2),eng,The House of Memory (Pluto's Snitch #2),nan
3,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)",[147734],US,,,B0056A00P4,B0056A00P4,,285263,12182387,1,4.04,4,"[{'count': '1057', 'name': 'to-read'}, {'count...",true,,,,,,,,This is the final tale in the bestselling auth...,"[{'author_id': '50873', 'role': ''}, {'author_...","[519546, 1295074, 21407416]","The Passion (Dark Visions, #3)",eng,"The Passion (Dark Visions, #3) This is the fin...",eng
4,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)",[811663],US,en-US,,B01BLJGA9S,B01BLJGA9S,,46079519,29074693,21,4.23,149,"[{'count': '598', 'name': 'to-read'}, {'count'...",true,,,,,,,,,"[{'author_id': '5360266', 'role': ''}]","[25515353, 20483269, 25650829, 18913492, 22578...","Prowled Darkness (Dante's Circle, #7)",eng,"Prowled Darkness (Dante's Circle, #7)",nan


In [19]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [20]:
df.shape

(258585, 30)

In [21]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [22]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",[189911],eng,,,8948723,7327624,7,4.03,140,"[{'count': '58', 'name': 'to-read'}, {'count':...",Hardcover,600,"Nelson Doubleday, Inc.",,,1987,Book Club Edition,Omnibus book club edition containing the Ladie...,"[{'author_id': '10333', 'role': ''}]","[19997, 828466, 1569323, 425389, 1176674, 2627..."
1,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],eng,1934876569,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216..."
2,The House of Memory (Pluto's Snitch #2),[1052227],eng,,,54143148,33394837,60,4.33,269,"[{'count': '54', 'name': 'currently-reading'},...",,318,,,,,,,"[{'author_id': '242185', 'role': ''}]",[]
3,"The Passion (Dark Visions, #3)",[147734],eng,,,285263,12182387,1,4.04,4,"[{'count': '1057', 'name': 'to-read'}, {'count...",,,,,,,,This is the final tale in the bestselling auth...,"[{'author_id': '50873', 'role': ''}, {'author_...","[519546, 1295074, 21407416]"
4,"Prowled Darkness (Dante's Circle, #7)",[811663],eng,,,46079519,29074693,21,4.23,149,"[{'count': '598', 'name': 'to-read'}, {'count'...",,,,,,,,,"[{'author_id': '5360266', 'role': ''}]","[25515353, 20483269, 25650829, 18913492, 22578..."


In [23]:
df.shape

(258585, 21)

In [24]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [25]:
df.shape

(210403, 21)

In [26]:
df['format'].value_counts().shape

(296,)

In [27]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['Hardcover', 'Paperback', '', 'ebook', 'Kindle Edition', 'Audiobook', 'Audio', 'Unknown Binding', 'Audio CD', 'MP3 CD', 'Mass Market Paperback', 'Audible Audio', 'Textbook Binding', 'online fiction', 'online short story', 'Audio Cassette', 'Novelty Book', 'Trade Paperback', 'Digital Comic', 'Preloaded Digital Audio Player', 'Nook', 'Leather Bound', 'Audiobook, Unabridged Audiobook (8hrs 55min)', 'nook book', 'hardcover', 'ebook &amp; paperback', 'Unbound', 'Library Binding', 'paperback', 'softcover', 'free online', 'e-book, digital format', 'Poche', 'Hardback', 'Online', 'Audiobook, Unabridged Audiobook (23hrs 30min)', 'Leather-Cloth', 'audio', 'Online Fiction', 'Boxed Set', 'Spiral-bound', 'Softcover', 'hardback', 'Kindle Edition, Paperback', 'Free Online Fiction', 'Online Fiction - Complete', 'Chapbook', 'Radio drama', 'iBooks', 'Podiobook', 'Paperback Box Set', "Publisher's Binding", 'free online fiction', 'P.D.F.', 'Special Edition Mass Market Paperback', 'Cloth', 'Kindle', 'Kobo 

In [28]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard physical books
    "Hardcover",
    "hardcover",
    "Hard Cover",
    "Hardback",
    "hardback",
    "Hardbound",
    "hardbound",
    "Paperback",
    "paperback",
    "Mass Market Paperback",
    "mass market paperback",
    "mass paperback",
    "Mass Market",
    "Trade Paperback",
    "trade Paperback",
    "trade paperback",
    "Trade paperback",
    "Trade paperback C format",
    "Premium Mass Market Paperback",
    "Special Edition Mass Market Paperback",
    "Perfect Paperback",
    "Softcover",
    "softcover",
    "soft cover",
    "Softback",
    "Hard Paperback",
    "Bound, Paperback",
    "UK Paperback",
    "Paperback (Large Print)",
    "Hardcover (Large Print)",
    "Hardcover - Deluxe Illustrated Edition",
    "Hardcover, Paper Dust Jacket",
    "Hardcover, Sewn Binding, Paper Dust Jacket",
    "Hardcover Slipcase",
    "Hardcover Boxed Set",
    "Hardcover book-plus product",
    "Hardcover with dust jacket",
    "hard cover - limited and/or lettered editions",
    "Hardcover, Paper Dust Jacket",
    "Paperback in Hard Sleeve",

    # Bindings / print variants
    "Textbook Binding",
    "Library Binding",
    "Library bound",
    "school binding",
    "School &amp; Library Binding",
    "Paperback School &amp; Library Binding",
    "Turtleback",
    "Leather Bound",
    "Leather-Cloth",
    "Cloth",
    "cloth",
    "Cloth Bound",
    "Fabric Bound",
    "Publisher's Binding",
    "Unbound",
    "Spiral-bound",
    "Chapbook",
    "chapbook",
    "Chapbook / Paperback",
    "Handbound chapbook",
    "handbound perfect binding",
    "Poche",
    "Broche",
    "broche",
    "Taschenbuch",
    "Capa Mole",
    "Livro de bolso",
    "Econo-clad",
    "Print on Demand",
    "Print",
    "A4 Paper",
    "paper",
    "Newsprint",
    "8.5&quot; x 7&quot; (half legal), B&amp;W, Saddle Stapled",
    "6 x 9 softcover; PDF (available seperately or bundled)",

    # Board / novelty / visual text
    "Board book",
    "Board Book",
    "Board book, Hardcover",
    "Novelty Book",
    "Visual Novel",

    # Box sets / collections / omnibus
    "Boxed Set",
    "Boxed set",
    "Boxed Collection",
    "Box Set",
    "Box",
    "Paperback Box Set",
    "Paperback (Boxed Set)",
    "Book Set",
    "Collection",
    "Part-In-Omnibus",
    "Science Fiction Book Club Omnibus",
    "Multiple Formats",
    "Multiple",
    "Book",

    # Ebooks / digital readable text
    "ebook",
    "eBook",
    "Ebook",
    "ebooks",
    "e-book",
    "E Book",
    "e book",
    "Kindle Edition",
    "Kindle edition",
    "Kindle",
    "Kindle eBook",
    "Kindle Edition, Nook",
    "Kobo ebook",
    "Kobo",
    "Sony Reader",
    "Sony eBook",
    "Nook",
    "Nook Edition",
    "Nook book",
    "nook book",
    "NOOKbook/ BN ebook",
    "iBooks",
    "Apple eBbook",
    "Digital",
    "digital",
    "PDF",
    "pdf",
    "P.D.F.",
    "epub",
    "PDF eBook",
    "pdf e-book",
    "Pdf e-book and Paperback",
    "PDF DIGITAL EDITION",
    "e-book, digital format",
    "Paperback; Digital (epub, mobi)",
    "Online eBook",
    "Ebook Novella",
    "Ebook Chapter Series",
    "ARC E-book",

    # Mixed print + ebook formats
    "ebook &amp; paperback",
    "Kindle Edition, Paperback",
    "Paperback, e-book",
    "Paperback &amp; eBook",
    "Paperback &amp; PDF",
    "Paperback, Kindle, Ebook, Audio",  # mixed, but contains readable editions
    "Paperback, eBook",
    "Paperback, Hardcover, E-book",
    "paperback, hardcover, e-book",
    "paperback, ebook",
    "Paperback and eBook",
    "Paperback and Kindle",
    "Paperback and Kindle Edition",
    "Paperback/Kindle",
    "Kindle, Paperback, Ebook",
    "Kindle Edition and Trade Paperback",
    "Kindle Edition, paperback",
    "ebook, paperback",
    "ebook, paperback, hardback",
    "ebook and paperback",
    "ebook and Paperback",
    "Ebook and paperback",
    "Ebook &amp; Paperback",
    "Ebook &amp; Paperback",
    "ebook and Print",
    "paperback and e-book",
    "paperback, e-book",
    "trade paperback and ebook",

    # Online fiction / web text
    "Online",
    "Online Fiction",
    "online fiction",
    "Online Fiction - Complete",
    "Free Online Fiction",
    "free online fiction",
    "free online",
    "free ebook",
    "Free Online",
    "Free Online Read",
    "Free online read",
    "Online Read",
    "Online Only",
    "Online Publication",
    "Online Novella",
    "Online Serial",
    "online serial",
    "Online short story",
    "online short story",
    "Web Fiction",
    "Web Serial",
    "Web Serial Novel",
    "Web Original",
    "Webpage",
    "web-published",
    "blog serial",
    "Serial Blog Posts",
    "Author Website",
    "Wattpad",
    "https://www.wattpad.com/story/3994065",
    "https://www.wattpad.com/story/49206761-mated-to-the-werewolf-king-now-completed",

    # Fanfiction / web fiction
    "fanfiction",
    "Fanfiction",
    "online fanfiction",

    # Comics / manga / graphic novels
    "Comic",
    "Comic Book",
    "Comics",
    "Graphic novel",
    "Graphic Novel",
    "Graphic Novel Paperback",
    "Kindle Graphic Novel",
    "Digital Comic",
    "Comixology",
    "Single Issue Comicbook",
    "online comic",
    "online graphic novel (membership/subscription)",
    "Manga",
    "Webtoon",
    "Webcomic",

    # ARC / prerelease readable
    "ARC",
    "Advance Reader's Copy",

    # Zines / magazines / short forms
    "Zine",
    "Magazine",
    "Magazine Novelette",
}

# 2. Define formats to drop
drop_formats = {
    # Missing / unknown / not useful
    "",
    "Unknown Binding",
    "none",
    "Other Format",

    # Audio
    "Audiobook",
    "audiobook",
    "Audio",
    "audio",
    "Audio CD",
    "audio CD",
    "Audo CD",
    "Audio CD - Abr",
    "Audio CD (Unabridged)",
    "Audio Cassette",
    "Audio Cassette Narrated by Madeleine L'Engle",
    "Audible Audio",
    "Audible",
    "Audible Download",
    "Audio Book",
    "Audio book",
    "audio book, podnovel, podcast",
    "Audiobook Unabridged",
    "Audiobook, Unabridged",
    "Audiobook, Unabridged Audiobook (1hrs 31min)",
    "Audiobook, Unabridged Audiobook (1hrs 58min)",
    "Audiobook, Unabridged Audiobook (8hrs 55min)",
    "Audiobook, Unabridged Audiobook (14hrs 40min)",
    "Audiobook, Unabridged Audiobook (15hrs 31min)",
    "Audiobook, Unabridged Audiobook (16hrs 19min)",
    "Audiobook, Unabridged Audiobook (17hrs 37min)",
    "Audiobook, Unabridged Audiobook (17hrs 9min)",
    "Audiobook, Unabridged Audiobook (19Hrs 45Min)",
    "Audiobook, Unabridged Audiobook (23hrs 30min)",
    "Audiobook - Audible Download",
    "audiobook/ebook",
    "eAudiobook",
    "Downloadable Audiobook",
    "Downloadable audio file",
    "Digital Audio",
    "Digtial Audio",
    "eAudio",
    "GraphicAudio",

    # MP3 / podcast / radio / playaway
    "MP3 CD",
    "MP3",
    "mp3",
    "MP3 Book",
    "MP3 Audiobook",
    "mp3 Audiobook Download",
    "OverDrive MP3 Audiobook",
    "Preloaded Digital Audio Player",
    "Pre-recorded MP3 player",
    "Playaway",
    "Playaway Audiobook",
    "Audio Playaway",
    "Podiobook",
    "Podcast",
    "Podcast Novel",
    "Podcast / Podiobook",
    "Podcast/Audiobook",
    "Audio Podcast",
    "Audio podcast",
    "Audio podcast.",
    "free podcast novel",
    "Radio drama",
    "radio drama theater",
    "Radio production",

    # Audio/video/app/software/disc
    "CD-ROM",
    "Android App",
    "Kindle Edition with Audio/Video",
    "audio/video",

    # Pirated / scanlation source
    "English scanlation",

    # Not available / not a book format
    "Pre-order",
    "Unpublished",

    # Weird/non-book product
    "Vinyl Cover",
    "Book and Toy",
    "Deck of Oracle Cards",
    "3-ring Binder",
    "Link",

    # Garbled / malformed
    "Paperback / shwmyz - rq`y",
}

# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      138855
other      71528
review        20
Name: count, dtype: int64

In [29]:
df = df[df["format_group"] == "text"].copy()

In [30]:
df["format_group"].value_counts()

format_group
text    138855
Name: count, dtype: int64

In [31]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",[189911],eng,,,8948723,7327624,7,4.03,140,"[{'count': '58', 'name': 'to-read'}, {'count':...",Hardcover,600,"Nelson Doubleday, Inc.",,,1987,Book Club Edition,Omnibus book club edition containing the Ladie...,"[{'author_id': '10333', 'role': ''}]","[19997, 828466, 1569323, 425389, 1176674, 2627...",Hardcover,text
1,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],eng,1934876569,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",Paperback,216,Seven Seas,3,3,2009,,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216...",Paperback,text
8,"Half Bad (Half Life, #1)",[493993],eng,0698143760,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",ebook,416,Viking Children's,4,3,2014,,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842...",ebook,text
10,"Healer's Touch (Hearts And Thrones, #4)",[709800],eng,,,43219002,23615331,11,3.75,112,"[{'count': '153', 'name': 'to-read'}, {'count'...",Kindle Edition,279,Amy Raby,23,11,2014,1st Edition,Marius believes himself to be an ordinary smal...,"[{'author_id': '6462472', 'role': ''}]","[15738275, 23557009, 17225973, 15750539, 23996...",Kindle Edition,text
12,"The Serpent and the Rose (War of the Rose, #1)",[152365],eng,0765313286,9780765313287,766938,780917,54,3.49,372,"[{'count': '419', 'name': 'to-read'}, {'count'...",Paperback,320,Tor Books,6,3,2007,,The beautiful Averil is heir to the Duchy of Q...,"[{'author_id': '410348', 'role': ''}]","[1616595, 2356304, 5432871, 9766146, 6058757, ...",Paperback,text


In [32]:
df.shape

(138855, 23)

In [33]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [34]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [35]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
0,"The Unschooled Wizard (Sun Wolf and Starhawk, ...",[189911],eng,,,8948723,7327624,7,4.03,140,"[{'count': '58', 'name': 'to-read'}, {'count':...",Hardcover,600,"Nelson Doubleday, Inc.",1987,Omnibus book club edition containing the Ladie...,"[{'author_id': '10333', 'role': ''}]","[19997, 828466, 1569323, 425389, 1176674, 2627..."
1,All's Fairy in Love and War (Avalon: Web of Ma...,[151854],eng,1934876569,9781934876565,701117,6066812,6,4.22,98,"[{'count': '515', 'name': 'to-read'}, {'count'...",Paperback,216,Seven Seas,2009,"To Kara's astonishment, she discovers that a p...","[{'author_id': '19158', 'role': ''}]","[948696, 439885, 274955, 12978730, 372986, 216..."
8,"Half Bad (Half Life, #1)",[493993],eng,0698143760,9780698143760,24802827,21401181,17,3.80,33,"[{'count': '1799', 'name': 'fantasy'}, {'count...",ebook,416,Viking Children's,2014,Wanted by no one.\nHunted by everyone.\nSixtee...,"[{'author_id': '7314532', 'role': ''}]","[15728807, 17182499, 15673520, 16081758, 17842..."
10,"Healer's Touch (Hearts And Thrones, #4)",[709800],eng,,,43219002,23615331,11,3.75,112,"[{'count': '153', 'name': 'to-read'}, {'count'...",Kindle Edition,279,Amy Raby,2014,Marius believes himself to be an ordinary smal...,"[{'author_id': '6462472', 'role': ''}]","[15738275, 23557009, 17225973, 15750539, 23996..."
12,"The Serpent and the Rose (War of the Rose, #1)",[152365],eng,0765313286,9780765313287,766938,780917,54,3.49,372,"[{'count': '419', 'name': 'to-read'}, {'count'...",Paperback,320,Tor Books,2007,The beautiful Averil is heir to the Duchy of Q...,"[{'author_id': '410348', 'role': ''}]","[1616595, 2356304, 5432871, 9766146, 6058757, ..."


In [36]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [37]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [38]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [39]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [40]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,505,Sorcery & Cecelia: or The Enchanted Chocolate ...,"[{'author_id': '36122', 'role': ''}, {'author_...",[[144396]],eng,"[0606328343, 0152046151, 1453254676, 044177559...","[9780606328340, 9780152046156, 9781453254677, ...","[15722552, 349441, 638073, 13637073, 535392, 1...","[Open Road Young Readers, Turtleback Books, HM...","[2012.0, 2004.0, 2003.0, 1988.0]",Two girls contend with sorcery in England's Re...,"[{'count': '12088', 'name': 'to-read'}, {'coun...","[[382870, 195381, 1103543, 97968, 34282, 22481...",1559,14607,4.01,350.0
1,555,The Renegades of Pern (Pern: Dragonriders of P...,"[{'author_id': '26', 'role': ''}]","[[180992, 186010]]",eng,"[0345340965, 0552130990, 0345419391, 226603886...","[9780345340962, 9780552130998, 9780345419392, ...","[1120865, 954390, 20082, 721681, 543330]","[Del Rey, Corgi, Presses Pocket]","[1989.0, 1990.0, 1997.0, 1991.0]","As long as the people of Pern could remember, ...","[{'count': '984', 'name': 'to-read'}, {'count'...","[[127583, 127582, 127581, 1103140, 61896, 7775...",154,14543,3.84,409.0
2,742,"Over Sea, Under Stone (The Dark is Rising, #1)","[{'author_id': '7308', 'role': ''}]",[[182290]],eng,"[0140303626, 141694964X, 0152590366, 002042785...","[9780140303629, 9781416949640, 9780152590369, ...","[896455, 210331, 3233552, 18277528, 471920, 13...","[Puffin Books, Simon Pulse, Harcourt Brace Jov...","[1968.0, 2007.0, 1979.0, 1989.0, 2012.0, 2004....","Simon, Jane and Barney Drew know that their ho...","[{'count': '9840', 'name': 'to-read'}, {'count...","[[24783, 377889, 694942, 774175, 36644, 312077...",1344,34540,3.85,368.0
3,764,The Elf Queen Of Shannara,"[{'author_id': '9629', 'role': ''}]","[[144399, 183568, 478867, 479257]]",eng,"[0099201313, 0345375580, 0345362993]","[9780099201311, 9780345375582, 9780345362995]","[1444165, 772925, 1501671]","[Legend, Del Rey]","[1993.0, 1992.0]","""Find the Elves and return them to the world o...","[{'count': '4718', 'name': 'to-read'}, {'count...","[[429883, 13881, 618245, 228990, 181494]]",18,574,4.02,405.0
4,807,Fog Magic,"[{'author_id': '171845', 'role': ''}]",[],eng,"[0140321632, 0812457749]","[9780140321630, 9780812457742]","[869797, 297891, 16104094, 7871075]","[Puffin Books, The Viking Press, Perfection Le...","[1986.0, 1969.0, 1943.0]","Originally published in 1943, this edition fea...","[{'count': '863', 'name': 'to-read'}, {'count'...","[[611853, 245104, 1807609, 1238356, 1833502, 9...",86,785,3.85,128.0


In [41]:
df = books_work_df.copy()

In [42]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,505,Sorcery & Cecelia: or The Enchanted Chocolate ...,"[{'author_id': '36122', 'role': ''}, {'author_...",[[144396]],eng,"[0606328343, 0152046151, 1453254676, 044177559...","[9780606328340, 9780152046156, 9781453254677, ...","[15722552, 349441, 638073, 13637073, 535392, 1...","[Open Road Young Readers, Turtleback Books, HM...","[2012.0, 2004.0, 2003.0, 1988.0]",Two girls contend with sorcery in England's Re...,"[{'count': '12088', 'name': 'to-read'}, {'coun...","[[382870, 195381, 1103543, 97968, 34282, 22481...",1559,14607,4.01,350.0
1,555,The Renegades of Pern (Pern: Dragonriders of P...,"[{'author_id': '26', 'role': ''}]","[[180992, 186010]]",eng,"[0345340965, 0552130990, 0345419391, 226603886...","[9780345340962, 9780552130998, 9780345419392, ...","[1120865, 954390, 20082, 721681, 543330]","[Del Rey, Corgi, Presses Pocket]","[1989.0, 1990.0, 1997.0, 1991.0]","As long as the people of Pern could remember, ...","[{'count': '984', 'name': 'to-read'}, {'count'...","[[127583, 127582, 127581, 1103140, 61896, 7775...",154,14543,3.84,409.0
2,742,"Over Sea, Under Stone (The Dark is Rising, #1)","[{'author_id': '7308', 'role': ''}]",[[182290]],eng,"[0140303626, 141694964X, 0152590366, 002042785...","[9780140303629, 9781416949640, 9780152590369, ...","[896455, 210331, 3233552, 18277528, 471920, 13...","[Puffin Books, Simon Pulse, Harcourt Brace Jov...","[1968.0, 2007.0, 1979.0, 1989.0, 2012.0, 2004....","Simon, Jane and Barney Drew know that their ho...","[{'count': '9840', 'name': 'to-read'}, {'count...","[[24783, 377889, 694942, 774175, 36644, 312077...",1344,34540,3.85,368.0
3,764,The Elf Queen Of Shannara,"[{'author_id': '9629', 'role': ''}]","[[144399, 183568, 478867, 479257]]",eng,"[0099201313, 0345375580, 0345362993]","[9780099201311, 9780345375582, 9780345362995]","[1444165, 772925, 1501671]","[Legend, Del Rey]","[1993.0, 1992.0]","""Find the Elves and return them to the world o...","[{'count': '4718', 'name': 'to-read'}, {'count...","[[429883, 13881, 618245, 228990, 181494]]",18,574,4.02,405.0
4,807,Fog Magic,"[{'author_id': '171845', 'role': ''}]",[],eng,"[0140321632, 0812457749]","[9780140321630, 9780812457742]","[869797, 297891, 16104094, 7871075]","[Puffin Books, The Viking Press, Perfection Le...","[1986.0, 1969.0, 1943.0]","Originally published in 1943, this edition fea...","[{'count': '863', 'name': 'to-read'}, {'count'...","[[611853, 245104, 1807609, 1238356, 1833502, 9...",86,785,3.85,128.0


In [43]:
df.shape

(87098, 17)

In [44]:
df.isna().sum()

work_id                      0
clean_title                  0
authors                      0
series_list                  0
language_code_final          0
isbn_list                    0
isbn13_list                  0
book_id_list                 0
publisher_list               0
publication_year_list        0
description               2996
popular_shelves              0
similar_books_list           0
text_reviews_count           0
ratings_count                0
average_rating               0
num_pages                11994
dtype: int64

In [45]:
df.shape

(87098, 17)

In [46]:
df = df[df["description"].notna()].copy()

In [47]:
df.isna().sum()

work_id                      0
clean_title                  0
authors                      0
series_list                  0
language_code_final          0
isbn_list                    0
isbn13_list                  0
book_id_list                 0
publisher_list               0
publication_year_list        0
description                  0
popular_shelves              0
similar_books_list           0
text_reviews_count           0
ratings_count                0
average_rating               0
num_pages                11276
dtype: int64

In [48]:
df.shape

(84102, 17)

In [49]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,505,Sorcery & Cecelia: or The Enchanted Chocolate ...,"[{'author_id': '36122', 'role': ''}, {'author_...",[[144396]],eng,"[0606328343, 0152046151, 1453254676, 044177559...","[9780606328340, 9780152046156, 9781453254677, ...","[15722552, 349441, 638073, 13637073, 535392, 1...","[Open Road Young Readers, Turtleback Books, HM...","[2012.0, 2004.0, 2003.0, 1988.0]",Two girls contend with sorcery in England's Re...,"[{'count': '12088', 'name': 'to-read'}, {'coun...","[[382870, 195381, 1103543, 97968, 34282, 22481...",1559,14607,4.01,350.0
1,555,The Renegades of Pern (Pern: Dragonriders of P...,"[{'author_id': '26', 'role': ''}]","[[180992, 186010]]",eng,"[0345340965, 0552130990, 0345419391, 226603886...","[9780345340962, 9780552130998, 9780345419392, ...","[1120865, 954390, 20082, 721681, 543330]","[Del Rey, Corgi, Presses Pocket]","[1989.0, 1990.0, 1997.0, 1991.0]","As long as the people of Pern could remember, ...","[{'count': '984', 'name': 'to-read'}, {'count'...","[[127583, 127582, 127581, 1103140, 61896, 7775...",154,14543,3.84,409.0
2,742,"Over Sea, Under Stone (The Dark is Rising, #1)","[{'author_id': '7308', 'role': ''}]",[[182290]],eng,"[0140303626, 141694964X, 0152590366, 002042785...","[9780140303629, 9781416949640, 9780152590369, ...","[896455, 210331, 3233552, 18277528, 471920, 13...","[Puffin Books, Simon Pulse, Harcourt Brace Jov...","[1968.0, 2007.0, 1979.0, 1989.0, 2012.0, 2004....","Simon, Jane and Barney Drew know that their ho...","[{'count': '9840', 'name': 'to-read'}, {'count...","[[24783, 377889, 694942, 774175, 36644, 312077...",1344,34540,3.85,368.0
3,764,The Elf Queen Of Shannara,"[{'author_id': '9629', 'role': ''}]","[[144399, 183568, 478867, 479257]]",eng,"[0099201313, 0345375580, 0345362993]","[9780099201311, 9780345375582, 9780345362995]","[1444165, 772925, 1501671]","[Legend, Del Rey]","[1993.0, 1992.0]","""Find the Elves and return them to the world o...","[{'count': '4718', 'name': 'to-read'}, {'count...","[[429883, 13881, 618245, 228990, 181494]]",18,574,4.02,405.0
4,807,Fog Magic,"[{'author_id': '171845', 'role': ''}]",[],eng,"[0140321632, 0812457749]","[9780140321630, 9780812457742]","[869797, 297891, 16104094, 7871075]","[Puffin Books, The Viking Press, Perfection Le...","[1986.0, 1969.0, 1943.0]","Originally published in 1943, this edition fea...","[{'count': '863', 'name': 'to-read'}, {'count'...","[[611853, 245104, 1807609, 1238356, 1833502, 9...",86,785,3.85,128.0


In [50]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [51]:
df.shape

(84102, 15)

In [52]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,505,Sorcery & Cecelia: or The Enchanted Chocolate ...,"[{'author_id': '36122', 'role': ''}, {'author_...",[[144396]],"[0606328343, 0152046151, 1453254676, 044177559...","[9780606328340, 9780152046156, 9781453254677, ...","[15722552, 349441, 638073, 13637073, 535392, 1...","[Open Road Young Readers, Turtleback Books, HM...",Two girls contend with sorcery in England's Re...,"[{'count': '12088', 'name': 'to-read'}, {'coun...","[[382870, 195381, 1103543, 97968, 34282, 22481...",1559,14607,4.01,350.0
1,555,The Renegades of Pern (Pern: Dragonriders of P...,"[{'author_id': '26', 'role': ''}]","[[180992, 186010]]","[0345340965, 0552130990, 0345419391, 226603886...","[9780345340962, 9780552130998, 9780345419392, ...","[1120865, 954390, 20082, 721681, 543330]","[Del Rey, Corgi, Presses Pocket]","As long as the people of Pern could remember, ...","[{'count': '984', 'name': 'to-read'}, {'count'...","[[127583, 127582, 127581, 1103140, 61896, 7775...",154,14543,3.84,409.0
2,742,"Over Sea, Under Stone (The Dark is Rising, #1)","[{'author_id': '7308', 'role': ''}]",[[182290]],"[0140303626, 141694964X, 0152590366, 002042785...","[9780140303629, 9781416949640, 9780152590369, ...","[896455, 210331, 3233552, 18277528, 471920, 13...","[Puffin Books, Simon Pulse, Harcourt Brace Jov...","Simon, Jane and Barney Drew know that their ho...","[{'count': '9840', 'name': 'to-read'}, {'count...","[[24783, 377889, 694942, 774175, 36644, 312077...",1344,34540,3.85,368.0
3,764,The Elf Queen Of Shannara,"[{'author_id': '9629', 'role': ''}]","[[144399, 183568, 478867, 479257]]","[0099201313, 0345375580, 0345362993]","[9780099201311, 9780345375582, 9780345362995]","[1444165, 772925, 1501671]","[Legend, Del Rey]","""Find the Elves and return them to the world o...","[{'count': '4718', 'name': 'to-read'}, {'count...","[[429883, 13881, 618245, 228990, 181494]]",18,574,4.02,405.0
4,807,Fog Magic,"[{'author_id': '171845', 'role': ''}]",[],"[0140321632, 0812457749]","[9780140321630, 9780812457742]","[869797, 297891, 16104094, 7871075]","[Puffin Books, The Viking Press, Perfection Le...","Originally published in 1943, this edition fea...","[{'count': '863', 'name': 'to-read'}, {'count'...","[[611853, 245104, 1807609, 1238356, 1833502, 9...",86,785,3.85,128.0


In [53]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [54]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [55]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [56]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [57]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [58]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [59]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '12088', 'name': 'to-read'}, {'count': '1649', 'name': 'fantasy'}, {'count': '400', 'name': 'ya'}, {'count': '360', 'name': 'historical-fiction'}, {'count': '333', 'name': 'favorites'}, {'count': '320', 'name': 'young-adult'}, {'count': '267', 'name': 'fiction'}, {'count': '241', 'name': 'romance'}, {'count': '215', 'name': 'historical'}, {'count': '157', 'name': 'magic'}, {'count': '133', 'name': 'mystery'}, {'count': '132', 'name': 'regency'}, {'count': '128', 'name': 'epistolary'}, {'count': '124', 'name': 'currently-reading'}, {'count': '124', 'name': 'series'}, {'count': '91', 'name': 'historical-fantasy'}, {'count': '74', 'name': 'teen'}, {'count': '69', 'name': 'ya-fantasy'}, {'count': '68', 'name': 'owned'}, {'count': '65', 'name': 'books-i-own'}, {'count': '52', 'name': 'kindle'}, {'count': '52', 'name': 'humor'}, {'count': '51', 'name': 'alternate-history'}, {'count': '49', 'name': 'sci-fi-fantasy'}, {'count': '48', 'name': 'fantasy-sci-fi'}, {'count': '45'

In [60]:
int_df = df[['popular_shelves']]

In [61]:
int_df.head()

,popular_shelves
0,"[{'count': '12088', 'name': 'to-read'}, {'coun..."
1,"[{'count': '984', 'name': 'to-read'}, {'count'..."
2,"[{'count': '9840', 'name': 'to-read'}, {'count..."
3,"[{'count': '4718', 'name': 'to-read'}, {'count..."
4,"[{'count': '863', 'name': 'to-read'}, {'count'..."


In [62]:
int_df.shape

(84102, 1)

In [63]:
int_df.to_csv('../data/intermediate/fantasy_shelves.csv', index=False)

In [64]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/fantasy_tag_mapping.csv')

In [65]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,86555714,83436
1,fantasy,fantasy,keep,5477890,72405
2,currently-reading,status,drop,3604616,66080
3,favorites,review,drop,2186642,42354
4,young-adult,young-adult,keep,1505028,25420
5,paranormal,paranormal,keep,1299553,48872
6,fiction,fiction,keep,1209087,48453
7,romance,romance,keep,1002387,39775
8,urban-fantasy,urban-fantasy,keep,786584,24819
9,series,noise,drop,769674,40373


In [66]:
vocab_df = shelves.copy()

In [67]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [68]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,86555714,83436
1,fantasy,fantasy,keep,5477890,72405
2,currently-reading,status,drop,3604616,66080
3,favorites,review,drop,2186642,42354
4,young-adult,young-adult,keep,1505028,25420


In [69]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [70]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [71]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '12088', 'name': 'to-read'}, {'coun...","[{'count': '1649', 'name': 'fantasy'}, {'count..."
1,"[{'count': '984', 'name': 'to-read'}, {'count'...","[{'count': '392', 'name': 'science-fiction'}, ..."
2,"[{'count': '9840', 'name': 'to-read'}, {'count...","[{'count': '666', 'name': 'fantasy'}, {'count'..."
3,"[{'count': '4718', 'name': 'to-read'}, {'count...","[{'count': '523', 'name': 'fantasy'}, {'count'..."
4,"[{'count': '863', 'name': 'to-read'}, {'count'...","[{'count': '66', 'name': 'fantasy'}, {'count':..."


In [72]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '12088', 'name': 'to-read'}, {'count': '1649', 'name': 'fantasy'}, {'count': '400', 'name': 'ya'}, {'count': '360', 'name': 'historical-fiction'}, {'count': '333', 'name': 'favorites'}, {'count': '320', 'name': 'young-adult'}, {'count': '267', 'name': 'fiction'}, {'count': '241', 'name': 'romance'}, {'count': '215', 'name': 'historical'}, {'count': '157', 'name': 'magic'}, {'count': '133', 'name': 'mystery'}, {'count': '132', 'name': 'regency'}, {'count': '128', 'name': 'epistolary'}, {'count': '124', 'name': 'currently-reading'}, {'count': '124', 'name': 'series'}, {'count': '91', 'name': 'historical-fantasy'}, {'count': '74', 'name': 'teen'}, {'count': '69', 'name': 'ya-fantasy'}, {'count': '68', 'name': 'owned'}, {'count': '65', 'name': 'books-i-own'}, {'count': '52', 'name': 'kindle'}, {'count': '52', 'name': 'humor'}, {'count': '51', 'name': 'alternate-history'}, {'count': '49', 'name': 'sci-fi-fantasy'}, {'count': '48', 'name': 'fantasy-sci-fi

In [73]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [74]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [75]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '734', 'name': 'science-fiction'}, {'count': '295', 'name': 'fantasy'}, {'count': '300', 'name': 'dragons'}, {'count': '172', 'name': 'fiction'}, {'count': '250', 'name': 'science-fantasy'}, {'count': '66', 'name': 'science'}, {'count': '28', 'name': 'adventure'}, {'count': '32', 'name': 'adult'}, {'count': '20', 'name': 'young-adult'}, {'count': '25', 'name': 'epic-fantasy'}, {'count': '11', 'name': 'space'}, {'count': '6', 'name': 'romance'}]


In [76]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '20', 'name': 'dragonlance'}
{'count': '25', 'name': 'fantasy'}
{'count': '8', 'name': 'fiction'}
{'count': '5', 'name': 'science-fantasy'}
{'count': '3', 'name': 'forgotten-realms'}
{'count': '3', 'name': 'adventure'}
{'count': '4', 'name': 'dragons'}
{'count': '4', 'name': 'epic-fantasy'}
{'count': '1', 'name': 'science-fiction'}
{'count': '1', 'name': 'nonfiction'}
{'count': '1', 'name': 'harry-potter'}
{'count': '2', 'name': 'adult'}
{'count': '1', 'name': 'family'}
{'count': '1', 'name': 'magic'}
{'count': '1', 'name': 'science'}
{'count': '1', 'name': 'classics'}


In [77]:
df.shape

(84102, 18)

In [78]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,505,Sorcery & Cecelia: or The Enchanted Chocolate Pot,"[{'author_id': '36122', 'role': ''}, {'author_...",[144396],"[0606328343, 0152046151, 1453254676, 044177559...","[9780606328340, 9780152046156, 9781453254677, ...","[15722552, 349441, 638073, 13637073, 535392, 1...","[Open Road Young Readers, Turtleback Books, HM...",Two girls contend with sorcery in England's Re...,"[{'count': '12088', 'name': 'to-read'}, {'coun...","[382870, 195381, 1103543, 97968, 34282, 224816...",1559,14607,4.01,350.0,"[36122, 36175]",0,"[{'count': '1709', 'name': 'fantasy'}, {'count..."
1,555,The Renegades of Pern,"[{'author_id': '26', 'role': ''}]","[180992, 186010]","[0345340965, 0552130990, 0345419391, 226603886...","[9780345340962, 9780552130998, 9780345419392, ...","[1120865, 954390, 20082, 721681, 543330]","[Del Rey, Corgi, Presses Pocket]","As long as the people of Pern could remember, ...","[{'count': '984', 'name': 'to-read'}, {'count'...","[127583, 127582, 127581, 1103140, 61896, 77752...",154,14543,3.84,409.0,[26],0,"[{'count': '734', 'name': 'science-fiction'}, ..."
2,742,"Over Sea, Under Stone","[{'author_id': '7308', 'role': ''}]",[182290],"[0140303626, 141694964X, 0152590366, 002042785...","[9780140303629, 9781416949640, 9780152590369, ...","[896455, 210331, 3233552, 18277528, 471920, 13...","[Puffin Books, Simon Pulse, Harcourt Brace Jov...","Simon, Jane and Barney Drew know that their ho...","[{'count': '9840', 'name': 'to-read'}, {'count...","[24783, 377889, 694942, 774175, 36644, 312077,...",1344,34540,3.85,368.0,[7308],0,"[{'count': '726', 'name': 'fantasy'}, {'count'..."
3,764,The Elf Queen Of Shannara,"[{'author_id': '9629', 'role': ''}]","[144399, 183568, 478867, 479257]","[0099201313, 0345375580, 0345362993]","[9780099201311, 9780345375582, 9780345362995]","[1444165, 772925, 1501671]","[Legend, Del Rey]","""Find the Elves and return them to the world o...","[{'count': '4718', 'name': 'to-read'}, {'count...","[429883, 13881, 618245, 228990, 181494]",18,574,4.02,405.0,[9629],0,"[{'count': '606', 'name': 'fantasy'}, {'count'..."
4,807,Fog Magic,"[{'author_id': '171845', 'role': ''}]",[],"[0140321632, 0812457749]","[9780140321630, 9780812457742]","[869797, 297891, 16104094, 7871075]","[Puffin Books, The Viking Press, Perfection Le...","Originally published in 1943, this edition fea...","[{'count': '863', 'name': 'to-read'}, {'count'...","[611853, 245104, 1807609, 1238356, 1833502, 93...",86,785,3.85,128.0,[171845],0,"[{'count': '73', 'name': 'fantasy'}, {'count':..."


In [79]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [80]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [81]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,505,Sorcery & Cecelia: or The Enchanted Chocolate Pot,"[36122, 36175]",[144396],"[0606328343, 0152046151, 1453254676, 044177559...","[9780606328340, 9780152046156, 9781453254677, ...","[15722552, 349441, 638073, 13637073, 535392, 1...","[Open Road Young Readers, Turtleback Books, HM...",Two girls contend with sorcery in England's Re...,"[382870, 195381, 1103543, 97968, 34282, 224816...",1559,14607,4.01,350.0,0,"[{'count': '1709', 'name': 'fantasy'}, {'count..."
1,555,The Renegades of Pern,[26],"[180992, 186010]","[0345340965, 0552130990, 0345419391, 226603886...","[9780345340962, 9780552130998, 9780345419392, ...","[1120865, 954390, 20082, 721681, 543330]","[Del Rey, Corgi, Presses Pocket]","As long as the people of Pern could remember, ...","[127583, 127582, 127581, 1103140, 61896, 77752...",154,14543,3.84,409.0,0,"[{'count': '734', 'name': 'science-fiction'}, ..."
2,742,"Over Sea, Under Stone",[7308],[182290],"[0140303626, 141694964X, 0152590366, 002042785...","[9780140303629, 9781416949640, 9780152590369, ...","[896455, 210331, 3233552, 18277528, 471920, 13...","[Puffin Books, Simon Pulse, Harcourt Brace Jov...","Simon, Jane and Barney Drew know that their ho...","[24783, 377889, 694942, 774175, 36644, 312077,...",1344,34540,3.85,368.0,0,"[{'count': '726', 'name': 'fantasy'}, {'count'..."
3,764,The Elf Queen Of Shannara,[9629],"[144399, 183568, 478867, 479257]","[0099201313, 0345375580, 0345362993]","[9780099201311, 9780345375582, 9780345362995]","[1444165, 772925, 1501671]","[Legend, Del Rey]","""Find the Elves and return them to the world o...","[429883, 13881, 618245, 228990, 181494]",18,574,4.02,405.0,0,"[{'count': '606', 'name': 'fantasy'}, {'count'..."
4,807,Fog Magic,[171845],[],"[0140321632, 0812457749]","[9780140321630, 9780812457742]","[869797, 297891, 16104094, 7871075]","[Puffin Books, The Viking Press, Perfection Le...","Originally published in 1943, this edition fea...","[611853, 245104, 1807609, 1238356, 1833502, 93...",86,785,3.85,128.0,0,"[{'count': '73', 'name': 'fantasy'}, {'count':..."


In [82]:
df.shape

(84102, 16)

In [83]:
df.to_csv('../data/intermediate/books-cleaned/fantasy.csv', index=False)